In [6]:
!pip install groq

In [7]:
!pip install langchain openai dotenv pydantic

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent 
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal

In [3]:
load_dotenv()

True

In [4]:
model = ChatOpenAI(model="gpt-4o-mini")

In [1]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-20b",
)

In [2]:
prompt = """
Understand the coding request:

For any request, identify the following :

- task type
- objective
- files involved 

Return the response in the following JSON format:
{
    "task_type": "type of task",
    "objective": "objective of the task",
    "files_involved": "list of files involved"
}

"""

In [3]:
request = "Identify the bug in the auth.py file where the exchange token is not getting generated"

In [4]:
response = model.invoke(prompt + request)
print(response.content)

{
    "task_type": "Bug Identification",
    "objective": "Identify the bug in auth.py where the exchange token is not getting generated",
    "files_involved": ["auth.py"]
}


In [8]:
class CodingRequest(BaseModel):
    """Structured representation of the coding request"""

    task_type: Literal[
        "implement",
        "debug",
        "review",
        "explain"
    ] = Field(description="Type of task to be performed")

    objective: str = Field(description="A concise description of what needs to be achieved")

    need_code_changes: bool = Field(
        description="Whether fulfilling this task required modifying the source code"
    )

    target_files: list[str] = Field(
        description="List of files that are relevant to the task, e.g. ['auth.py', 'models.py']"
    )

In [9]:
structured_output_model = model.with_structured_output(CodingRequest)

In [10]:
result = structured_output_model.invoke("""
    Fix the login bug in auth.py. Expired sessions currently produce HTTP 500.

""")

In [11]:
print(type(result))

<class '__main__.CodingRequest'>


In [12]:
CodingRequest.model_json_schema()

{'description': 'Structured representation of the coding request',
 'properties': {'task_type': {'description': 'Type of task to be performed',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A concise description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'Whether fulfilling this task required modifying the source code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'description': "List of files that are relevant to the task, e.g. ['auth.py', 'models.py']",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes', 'target_files'],
 'title': 'CodingRequest',
 'type': 'object'}